# Signature Verification - Kaggle Training

This notebook validates the manifest, runs a SupCon smoke test, trains the full SupCon model, evaluates it with one reference signature, and plots per-dataset results.

The notebook does not silently prune missing files or evaluate a checkpoint that was not trained.

In [ ]:
%cd /kaggle/working
!rm -rf sigver
!git clone https://github.com/yogeshY0/signa.git sigver
%cd /kaggle/working/sigver
!pip install -q timm opencv-python-headless matplotlib

import csv
import os
from collections import Counter
from pathlib import Path

DATA = "/kaggle/input/datasets/yogeshbaral/dataset-to-train-version-6"
SOURCE_MANIFEST = os.path.join(DATA, "manifest.csv")
MANIFEST = "/kaggle/working/manifest_clean.csv"
assert os.path.isfile(SOURCE_MANIFEST), f"manifest.csv missing at {SOURCE_MANIFEST}"
print("DATA =", DATA)
print("SOURCE_MANIFEST =", SOURCE_MANIFEST)


In [ ]:
!python -m preprocessing.prune_manifest --manifest "{SOURCE_MANIFEST}" --image-root "{DATA}" --output "{MANIFEST}"

rows = list(csv.DictReader(Path(MANIFEST).open()))
missing = [row["image_path"] for row in rows if not (Path(DATA) / row["image_path"]).is_file()]
by_split = {split: {row["signer_id"] for row in rows if row["split"] == split} for split in ("train", "val", "test")}

print("Rows:", len(rows))
print("Splits:", Counter(row["split"] for row in rows))
print("Datasets:", Counter(row["signer_id"].rsplit("_", 1)[0] for row in rows))
print("Missing after pruning:", len(missing))
print("Signer counts:", {name: len(value) for name, value in by_split.items()})
assert not missing, f"Manifest still contains missing files: {missing[:5]}"
assert not (by_split["train"] & by_split["val"]), "train/val signer overlap"
assert not (by_split["train"] & by_split["test"]), "train/test signer overlap"
assert not (by_split["val"] & by_split["test"]), "val/test signer overlap"


## Smoke test

Run this before the long job. It validates CUDA, image loading, batching, SupCon, checkpoint creation, and one-reference validation.

In [ ]:
!rm -rf /kaggle/working/smoke_supcon
!bash -o pipefail -c 'python -m training.train_siamese --manifest "$1" --image-root "$2" --output-dir /kaggle/working/smoke_supcon --device cuda --workers 2 --phase1-epochs 1 --phase2-epochs 1 --signers-per-batch 4 --genuine-per-signer 2 --forgeries-per-signer 2 --dataset-mix equal --references 1 --loss supcon --temperature 0.1 2>&1 | tee /kaggle/working/smoke_supcon.log' _ "{MANIFEST}" "{DATA}"
assert os.path.isfile("/kaggle/working/smoke_supcon/best_model.pt"), "Smoke test did not create a checkpoint"
print("SupCon smoke test passed")


## Full SupCon training

The validation threshold and checkpoint selection use one genuine reference per signer, matching deployment.

In [ ]:
!rm -rf /kaggle/working/v6_supcon
!bash -o pipefail -c 'python -m training.train_siamese --manifest "$1" --image-root "$2" --output-dir /kaggle/working/v6_supcon --device cuda --workers 2 --phase1-epochs 5 --phase2-epochs 30 --dataset-mix equal --references 1 --patience 6 --loss supcon --temperature 0.1 --seed 42 2>&1 | tee /kaggle/working/v6_supcon.log' _ "{MANIFEST}" "{DATA}"
assert os.path.isfile("/kaggle/working/v6_supcon/best_model.pt"), "Full training did not create a checkpoint"


In [ ]:
!python -m training.analyze_log /kaggle/working/v6_supcon.log --plot /kaggle/working/curves_v6_supcon.png
from IPython.display import Image, display
display(Image("/kaggle/working/curves_v6_supcon.png"))


In [ ]:
%%bash
set -euo pipefail
MANIFEST="/kaggle/working/manifest_clean.csv"
DATA="/kaggle/input/datasets/yogeshbaral/dataset-to-train-version-6"
CKPT="/kaggle/working/v6_supcon/best_model.pt"
python -m training.evaluate --checkpoint "$CKPT" --manifest "$MANIFEST" --image-root "$DATA" --device cuda --workers 2 --references 1 --bootstrap 1000 > /kaggle/working/eval_v6_supcon.json
cat /kaggle/working/eval_v6_supcon.json


In [ ]:
import json
import matplotlib.pyplot as plt

with open("/kaggle/working/eval_v6_supcon.json") as file:
    result = json.load(file)
test = result["test"]
print("Validation skilled EER:", round(result["validation_eer_skilled"] * 100, 2), "%")
print("Test skilled EER:", round(test["eer_skilled"] * 100, 2), "%")
print("Skilled FAR:", round(test["far_skilled_at_threshold"] * 100, 2), "%")
print("FRR:", round(test["frr_at_threshold"] * 100, 2), "%")
print("Random FAR:", round(test["far_random_at_threshold"] * 100, 2), "%")

datasets = sorted(result["test_per_dataset"])
values = [result["test_per_dataset"][name]["eer_skilled"] * 100 for name in datasets]
plt.figure(figsize=(9, 5))
plt.bar(datasets, values)
plt.ylabel("Skilled-forgery EER (%)")
plt.title("V6 SupCon per-dataset skilled-forgery EER (1-reference)")
plt.grid(axis="y", alpha=0.3)
plt.show()
